# Stage 1 — V2 V-JEPA 2.1-B

Uses the team-provided fixed `train.csv` / `val.csv` directly.
No manifest/split module is used.

Set the local V-JEPA 2 source/checkpoint paths in `configs/stage1/vjepa2_1_b.yaml`.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

try:
    import blackbox_detection  # noqa: F401
except ModuleNotFoundError:
    _root = Path.cwd()
    while _root != _root.parent and not (_root / "pyproject.toml").is_file():
        _root = _root.parent
    sys.path.insert(0, str(_root / "src"))

from blackbox_detection.utils import seed_everything, setup_logger

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import json

from blackbox_detection.stage1.dataset import (
    Stage1VideoDataset,
    build_dataloader,
    video_batch_adapter,
)
from blackbox_detection.stage1.evaluator import (
    AggregationConfig,
    Stage1Evaluator,
    probabilities_to_labels,
    save_predictions,
    search_best_threshold,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms
from blackbox_detection.utils import load_checkpoint, stage1_score

logger = setup_logger("stage1.vjepa2_1_b")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Team-provided fixed split CSVs.
# Change DATA_DIR / VIDEO_ROOT only if your teammate stores them elsewhere.
DATA_DIR = REPO_ROOT / "data" / "stage1"
TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"   # reserved for final holdout evaluation; not used for tuning

# Relative video_path values inside CSV are resolved against VIDEO_ROOT.
VIDEO_ROOT = REPO_ROOT

print("repo    :", REPO_ROOT)
print("configs :", CONFIG_DIR)
print("outputs :", OUTPUT_ROOT)
print("train   :", TRAIN_CSV)
print("val     :", VAL_CSV)
print("test    :", TEST_CSV, "(not loaded in training notebooks)")

## 3. Config

In [ ]:
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))
MODEL_NAME = CONFIG["model"]["name"]
ADAPTER = video_batch_adapter()

params = CONFIG["model"]["params"]
if params.get("source_root") is None:
    raise ValueError(
        "Set model.params.source_root in configs/stage1/vjepa2_1_b.yaml "
        "to a local clone of facebookresearch/vjepa2."
    )
if params.get("checkpoint_path") is None and not params.get("allow_download", False):
    raise ValueError(
        "Set model.params.checkpoint_path to vjepa2_1_vitb_dist_vitG_384.pt "
        "or explicitly enable allow_download."
    )

SEED = int(CONFIG["train"]["seed"])
RUN_DIR = REPO_ROOT / CONFIG["train"]["output_dir"]
RUN_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED, deterministic=False)
print(MODEL_NAME, "->", RUN_DIR)
print(json.dumps(params, indent=2))

## 4. Fixed CSV data

In [ ]:
REQUIRED_COLUMNS = {"video_path", "label", "video_id", "dataset"}
ALLOWED_LABELS = {"ORIGINAL", "RERECORDED"}

def load_stage1_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}")

    frame = pd.read_csv(path)
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    frame = frame.copy()
    frame["label"] = frame["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(frame["label"]) - ALLOWED_LABELS)
    if unknown:
        raise ValueError(f"{path.name} has unknown labels: {unknown}")

    def resolve_video_path(value: str) -> str:
        p = Path(str(value))
        if not p.is_absolute():
            p = VIDEO_ROOT / p
        return str(p.resolve())

    frame["video_path"] = frame["video_path"].map(resolve_video_path)
    frame["video_id"] = frame["video_id"].astype(str)
    frame["dataset"] = frame["dataset"].astype(str)

    if frame["video_id"].duplicated().any():
        duplicated = frame.loc[frame["video_id"].duplicated(), "video_id"].head().tolist()
        raise ValueError(f"{path.name} has duplicated video_id values: {duplicated}")

    missing_files = [p for p in frame["video_path"] if not Path(p).is_file()]
    if missing_files:
        raise FileNotFoundError(
            f"{path.name}: {len(missing_files)} video file(s) do not exist. "
            f"First examples: {missing_files[:3]}"
        )
    return frame.reset_index(drop=True)

train_df = load_stage1_csv(TRAIN_CSV)
val_df = load_stage1_csv(VAL_CSV)

overlap = set(train_df["video_id"]) & set(val_df["video_id"])
if overlap:
    raise ValueError(f"train/val video_id leakage detected: {sorted(overlap)[:5]}")

print("train:", len(train_df), train_df["label"].value_counts().to_dict())
print("val  :", len(val_df), val_df["label"].value_counts().to_dict())
print("datasets(train):", train_df["dataset"].value_counts().to_dict())
print("datasets(val)  :", val_df["dataset"].value_counts().to_dict())

## 5. Model

In [ ]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG["model"]["finetune_mode"],
    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),
    **params,
)

print("blocks:", len(model.blocks), "| feature dim:", model.feature_dim)
print("parameters:", count_parameters(model))
print("load report:", json.dumps(model.load_report["weights"], indent=2, default=str))
assert "random init" not in str(model.load_report["weights"]["source"])

### 5.1 Datasets and loaders

In [ ]:
video_config = CONFIG["data"]
augmentation_config = CONFIG["augmentation"]

preprocessing = model.preprocessing()
print("checkpoint preprocessing:", preprocessing)

train_transform, val_transform = build_video_transforms(
    crop_size=int(preprocessing["input_size"]),
    mean=tuple(preprocessing["mean"]),
    std=tuple(preprocessing["std"]),
    train_config=ClipAugmentConfig(
        crop_size=int(preprocessing["input_size"]),
        scale_range=tuple(augmentation_config["scale_range"]),
        ratio_range=tuple(augmentation_config["ratio_range"]),
        hflip_prob=float(augmentation_config["hflip_prob"]),
        brightness=float(augmentation_config["brightness"]),
        contrast=float(augmentation_config["contrast"]),
        perspective_prob=float(augmentation_config["perspective_prob"]),
        perspective_scale=float(augmentation_config["perspective_scale"]),
    ),
)

train_dataset = Stage1VideoDataset(
    train_df,
    clip_sampler=build_clip_sampler(
        train=True,
        num_frames=int(video_config["num_frames"]),
        strides=video_config["train_strides"],
        num_clips=int(video_config["train_num_clips"]),
    ),
    transform=train_transform,
    on_error="zero",
    deterministic=False,
)
val_dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(
        train=False,
        num_frames=int(video_config["num_frames"]),
        val_stride=int(video_config["val_stride"]),
        num_clips=int(video_config["val_num_clips"]),
    ),
    transform=val_transform,
    on_error="zero",
    deterministic=True,
)

train_loader = build_dataloader(
    train_dataset,
    batch_size=int(video_config["batch_size"]),
    shuffle=True,
    num_workers=int(video_config["num_workers"]),
    seed=SEED,
    drop_last=True,
)
val_loader = build_dataloader(
    val_dataset,
    batch_size=int(video_config["val_batch_size"]),
    shuffle=False,
    num_workers=int(video_config["num_workers"]),
    seed=SEED,
)

batch = next(iter(train_loader))
print("clip batch:", tuple(batch["pixels"].shape), "| labels:", batch["label"].tolist())

In [ ]:
static = np.repeat(np.full((1, 240, 320, 3), 128, dtype=np.uint8), 8, axis=0)
augmented = train_transform(static, np.random.default_rng(0))
spread = float(augmented.mean(dim=(0, 2, 3)).std())
print("per-frame mean spread on a static clip:", spread)
assert spread < 1e-5, "augmentation is not clip-consistent"

## 6. Training

In [ ]:
train_config = CONFIG["train"]

trainer_config = TrainConfig(
    epochs=int(train_config["epochs"]),
    learning_rate=float(train_config["learning_rate"]),
    head_learning_rate=(
        float(train_config["head_learning_rate"])
        if train_config.get("head_learning_rate") is not None
        else None
    ),
    weight_decay=float(train_config["weight_decay"]),
    warmup_ratio=float(train_config["warmup_ratio"]),
    grad_accum_steps=int(train_config["grad_accum_steps"]),
    max_grad_norm=float(train_config["max_grad_norm"]),
    amp=bool(train_config["amp"]),
    label_smoothing=float(train_config.get("label_smoothing", 0.0)),
    early_stopping_patience=int(train_config["early_stopping_patience"]),
    eval_every=int(train_config["eval_every"]),
    seed=SEED,
    output_dir=RUN_DIR,
    model_name=MODEL_NAME,
    wandb_enabled=False,
)

trainer = Stage1Trainer(
    model,
    trainer_config,
    adapter=ADAPTER,
    aggregation=AggregationConfig(
        frame_method=CONFIG["evaluation"]["aggregation"]["frame_method"],
        video_method=CONFIG["evaluation"]["aggregation"]["video_method"],
    ),
    model_config={"name": MODEL_NAME, "params": CONFIG["model"]["params"]},
)

print("device:", trainer.device, "| amp:", trainer.amp)
outcome = trainer.fit(train_loader, val_loader)
print(
    f"best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} "
    f"at threshold {outcome.best_threshold:.3f}"
)

## 7. Validation

In [ ]:
load_checkpoint(
    RUN_DIR / "best.pt",
    model=model,
    map_location=trainer.device,
    restore_rng_state=False,
)

evaluator = Stage1Evaluator(
    model,
    ADAPTER,
    device=trainer.device,
    amp=trainer.amp,
    aggregation=trainer.aggregation,
)
result, units = evaluator.evaluate(val_loader, return_units=True)

print(f"Macro-F1            : {result.macro_f1:.4f}")
print(f"Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}")
print(f"optimal threshold   : {result.threshold:.4f}")
print(f"class-wise F1       : {result.per_class_f1}")
print(f"per-dataset Macro-F1: {result.dataset_scores}")
print(f"videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)")

In [ ]:
labels = result.predictions["label"].tolist()
probabilities = result.predictions["prob_rerecorded"].to_numpy()

sweep = pd.DataFrame({"threshold": np.round(np.arange(0.05, 1.0, 0.05), 2)})
sweep["macro_f1"] = [
    stage1_score(labels, probabilities_to_labels(probabilities, threshold))
    for threshold in sweep["threshold"]
]
display(sweep.set_index("threshold").T)

best_threshold, best_score = search_best_threshold(labels, probabilities)
print(f"searched threshold {best_threshold:.4f} -> Macro-F1 {best_score:.4f}")

## 8. Save

In [ ]:
save_predictions(result.predictions, RUN_DIR / "val_predictions.csv")
outcome.history.to_csv(RUN_DIR / "history.csv", index=False)

summary = {
    "model_name": MODEL_NAME,
    "val_macro_f1": float(result.macro_f1),
    "val_macro_f1_at_0.5": float(result.macro_f1_at_default),
    "best_threshold": float(result.threshold),
    "per_class_f1": result.per_class_f1,
    "best_epoch": int(outcome.best_epoch),
    "num_val_videos": int(result.num_videos),
    "preprocessing": dict(model.preprocessing()),
}
(RUN_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, default=str),
    encoding="utf-8",
)
print("saved to:", RUN_DIR)